In [1076]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t

from src.experiment_runner.models import ExperimentRun, RequestRecord

# Loading functions

In [1077]:
def load_run(path: Path) -> ExperimentRun:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    reqs = [RequestRecord(**r) for r in data["requests"]]
    data["requests"] = reqs

    return ExperimentRun(**data)

In [1078]:
def compute_wa(run: ExperimentRun) -> float:
    replicas = [
        len(r.upstream.get("sockets", [])) or 1
        for r in run.requests
    ]
    return float(np.mean(replicas))

In [1079]:
def extract_row(run: ExperimentRun, group: str) -> dict:
    balancer = run.factors.get("balancer")
    replication = run.factors.get("replication") or "none"
    seed = run.factors.get("seed", -1)

    latency = run.summary["overall"]["latency_ms"]

    p99 = latency["p99"]
    p95 = latency["p95"]
    wa = compute_wa(run)

    return {
        "run_id": run.run_id,
        "seed": seed,
        "algorithm": balancer,
        "strategy": replication,
        "group": group,
        "p99": p99,
        "p95": p95,
        "wa": wa,
    }

In [1080]:
def collect_rows(root: Path) -> pd.DataFrame:
    rows = []

    for file in root.rglob("*.json"):
        if "baseline" in file.parts:
            group = "baseline"
        elif "non_adaptive" in file.parts:
            group = "non_adaptive"
        elif "adaptive" in file.parts:
            group = "adaptive"
        else:
            continue

        run = load_run(file)
        row = extract_row(run, group)

        # batch_id = timestamp папки эксперимента
        row["batch_id"] = file.parents[1].name
        row["file_name"] = file.name

        rows.append(row)

    return pd.DataFrame(rows)

In [1081]:
def aggregate(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df
        .groupby(["algorithm", "strategy", "group"], as_index=False)
        .agg({
            "p99": "mean",
            "wa": "mean",
        })
    )

In [1082]:
def compute_scores(df: pd.DataFrame) -> pd.DataFrame:
    results = []

    for group in ["non_adaptive", "adaptive"]:
        repl = df[df.group == group].copy()
        base = df[df.group == "baseline"].copy()

        if repl.empty:
            continue

        # Парное сравнение по algorithm + seed
        merged = repl.merge(
            base,
            on=["algorithm", "seed"],
            suffixes=("", "_baseline"),
        )

        if merged.empty:
            continue

        merged["p99_improvement_percent"] = (
                (merged["p99_baseline"] - merged["p99"]) / merged["p99_baseline"] * 100
        )

        merged["p95_improvement_percent"] = (
                (merged["p95_baseline"] - merged["p95"]) / merged["p95_baseline"] * 100
        )

        merged["wa_ratio"] = merged["wa"] / merged["wa_baseline"]

        merged["efficiency"] = (
                merged["p99_improvement_percent"] / merged["wa_ratio"]
        )

        merged["group"] = group

        agg = (
            merged
            .groupby(["algorithm", "strategy", "group"], as_index=False)
            .agg(
                p99_improvement_percent=("p99_improvement_percent", "mean"),
                p99_median=("p99_improvement_percent", "median"),
                p99_min=("p99_improvement_percent", "min"),
                p99_max=("p99_improvement_percent", "max"),
                p99_std=("p99_improvement_percent", "std"),

                p95_improvement_percent=("p95_improvement_percent", "mean"),
                p95_median=("p95_improvement_percent", "median"),
                p95_min=("p95_improvement_percent", "min"),
                p95_max=("p95_improvement_percent", "max"),
                p95_std=("p95_improvement_percent", "std"),

                wa=("wa_ratio", "mean"),
                efficiency=("efficiency", "mean"),
                experiments_count=("seed", "count"),
                positive_p99_rate=("p99_improvement_percent", lambda s: float((s > 0).mean())),
                positive_p95_rate=("p95_improvement_percent", lambda s: float((s > 0).mean())),
            )
        )

        n = agg["experiments_count"]
        t_crit_95 = t.ppf(0.975, df=n-1)
        t_crit_99 = t.ppf(0.995, df=n-1)

        ci_half_p99_95 = t_crit_95 * agg["p99_std"].fillna(0.0) / np.sqrt(n)
        low_p99 = agg["p99_improvement_percent"] - ci_half_p99_95
        high_p99 = agg["p99_improvement_percent"] + ci_half_p99_95
        agg["p99_ci95_range"] = (
                low_p99.round(2).astype(str) + " – " + high_p99.round(2).astype(str)
        )

        ci_half_p95_95 = t_crit_95 * agg["p95_std"].fillna(0.0) / np.sqrt(n)
        low_p95 = agg["p95_improvement_percent"] - ci_half_p95_95
        high_p95 = agg["p95_improvement_percent"] + ci_half_p95_95
        agg["p95_ci95_range"] = (
                low_p95.round(2).astype(str) + " – " + high_p95.round(2).astype(str)
        )

        agg["score"] = (
            agg["p99_improvement_percent"] / agg["wa"]
        )

        results.append(agg)

    if not results:
        return pd.DataFrame()

    final = pd.concat(results, ignore_index=True)
    final = final.sort_values(
        by=["algorithm", "group", "p99_improvement_percent"],
        ascending=[True, True, False],
    )
    return final

In [1083]:
def build_paired_comparison(df: pd.DataFrame, group: str = "non_adaptive") -> pd.DataFrame:
    repl = df[df.group == group].copy()
    base = df[df.group == "baseline"].copy()

    merged = repl.merge(
        base,
        on=["algorithm", "seed"],
        suffixes=("", "_baseline"),
    )

    if merged.empty:
        return pd.DataFrame()

    merged["delta_p99"] = merged["p99_baseline"] - merged["p99"]
    merged["delta_p95"] = merged["p95_baseline"] - merged["p95"]

    merged["p99_improvement_percent"] = (
            merged["delta_p99"] / merged["p99_baseline"] * 100
    )
    merged["p95_improvement_percent"] = (
            merged["delta_p95"] / merged["p95_baseline"] * 100
    )

    cols = [
        "algorithm",
        "strategy",
        "seed",
        "p99_baseline",
        "p99",
        "delta_p99",
        "p99_improvement_percent",
        "p95_baseline",
        "p95",
        "delta_p95",
        "p95_improvement_percent",
        "wa_baseline",
        "wa",
    ]
    return merged[cols].sort_values(["algorithm", "strategy", "seed"])

In [1084]:
def analyze_all(root: Path):
    df = collect_rows(root)
    paired = build_paired_comparison(df, group="non_adaptive")

    if df.empty:
        raise ValueError("Нет данных")

    return compute_scores(df), paired

# Results

In [1085]:
# rows = []
#
# for i in (4, 6, 7, 8, 9):
#     GROUP_DIR = Path(f"../assets/experiments/error_new{i}")
#     res, paired = analyze_all(GROUP_DIR)
#
#     rows.append({
#         "group": i,
#         "p99_improvement_percent": res['p99_improvement_percent'].iloc[0],
#         "p99_min": res['p99_min'].iloc[0],
#         "p99_max": res['p99_max'].iloc[0],
#         "p99_ci95_range": res['p99_ci95_range'].iloc[0],
#         "wa": res['wa'].iloc[0],
#         "score": res['score'].iloc[0],
#     })
#
# df = pd.DataFrame(rows)
#
# print(df)
# df

In [1086]:
GROUP_DIR = Path(f"../assets/experiments/error_new12")
res, paired = analyze_all(GROUP_DIR)

res

,algorithm,strategy,group,p99_improvement_percent,p99_median,p99_min,p99_max,p99_std,p95_improvement_percent,p95_median,...,p95_max,p95_std,wa,efficiency,experiments_count,positive_p99_rate,positive_p95_rate,p99_ci95_range,p95_ci95_range,score
1,topsis,hedged,adaptive,11.759459,11.93076,-0.609095,33.470036,6.971736,-3.040925,-3.216895,...,5.898623,4.885055,1.60170,7.329010,20,0.95,0.30,8.5 – 15.02,-5.33 – -0.75,7.341861
0,topsis,hedged,non_adaptive,12.393419,11.70430,-1.745400,30.865495,6.017736,-4.275394,-4.673223,...,5.038350,4.452836,1.65045,7.501123,20,0.95,0.15,9.58 – 15.21,-6.36 – -2.19,7.509115


In [1087]:
print(paired)
paired

   algorithm strategy   seed  p99_baseline           p99    delta_p99  \
0     topsis   hedged  10000  18847.446196  16329.818913  2517.627283   
1     topsis   hedged  10001  18644.626276  15535.519600  3109.106676   
2     topsis   hedged  10002  18355.020834  15130.693715  3224.327119   
3     topsis   hedged  10003  18288.419015  15693.164886  2595.254129   
4     topsis   hedged  10004  23505.104758  16250.137786  7254.966972   
5     topsis   hedged  10005  18743.502248  17319.812475  1423.689773   
6     topsis   hedged  10006  18555.343414  16533.016193  2022.327221   
7     topsis   hedged  10007  18264.432896  15399.537376  2864.895520   
8     topsis   hedged  10008  18114.893824  16050.197173  2064.696651   
9     topsis   hedged  10009  18903.480874  17105.266651  1798.214223   
10    topsis   hedged  10010  18642.045928  16653.788561  1988.257367   
11    topsis   hedged  10011  18553.235548  16104.256876  2448.978672   
12    topsis   hedged  10012  19888.461574  17090.9

,algorithm,strategy,seed,p99_baseline,p99,delta_p99,p99_improvement_percent,p95_baseline,p95,delta_p95,p95_improvement_percent,wa_baseline,wa
0,topsis,hedged,10000,18847.446196,16329.818913,2517.627283,13.357923,11253.152420,11429.320765,-176.168345,-1.565502,1.0,1.6546
1,topsis,hedged,10001,18644.626276,15535.519600,3109.106676,16.675618,10763.841455,10837.572415,-73.730960,-0.684987,1.0,1.6354
2,topsis,hedged,10002,18355.020834,15130.693715,3224.327119,17.566459,11078.520310,11417.029110,-338.508800,-3.055542,1.0,1.6354
3,topsis,hedged,10003,18288.419015,15693.164886,2595.254129,14.190697,10553.196365,11475.838470,-922.642105,-8.742774,1.0,1.6790
4,topsis,hedged,10004,23505.104758,16250.137786,7254.966972,30.865495,10895.149210,11286.196230,-391.047020,-3.589185,1.0,1.6696
5,topsis,hedged,10005,18743.502248,17319.812475,1423.689773,7.595644,11974.747790,11641.723870,333.023920,2.781052,1.0,1.6152
6,topsis,hedged,10006,18555.343414,16533.016193,2022.327221,10.898894,11170.441735,11468.865045,-298.423310,-2.671544,1.0,1.6896
7,topsis,hedged,10007,18264.432896,15399.537376,2864.895520,15.685653,10054.782750,10651.789845,-597.007095,-5.937543,1.0,1.6592
8,topsis,hedged,10008,18114.893824,16050.197173,2064.696651,11.397785,10510.795725,11477.237470,-966.441745,-9.194753,1.0,1.6234
9,topsis,hedged,10009,18903.480874,17105.266651,1798.214223,9.512609,11634.256065,11799.251490,-164.995425,-1.418186,1.0,1.6028


In [1088]:
def collect_request_level(root: Path) -> pd.DataFrame:
    rows = []

    for file in root.rglob("*.json"):
        # определяем группу
        if "baseline" in file.parts:
            group = "baseline"
        elif "non_adaptive" in file.parts:
            group = "non_adaptive"
        elif "adaptive" in file.parts:
            group = "adaptive"
        else:
            continue

        run = load_run(file)

        algorithm = run.factors.get("balancer")
        strategy = run.factors.get("replication") or "none"

        for r in run.requests:
            rows.append({
                "algorithm": algorithm,
                "strategy": strategy,
                "group": group,
                "ok": r.ok,
            })

    return pd.DataFrame(rows)

In [1089]:
df_req = collect_request_level(GROUP_DIR)

error_stats = (
    df_req
    .groupby(["algorithm", "strategy", "group"])["ok"]
    .agg(
        total="count",
        success="sum"
    )
)

error_stats["errors"] = error_stats["total"] - error_stats["success"]

error_stats["error_rate"] = (
        error_stats["errors"] / error_stats["total"]
)

error_stats["error_rate_percent"] = (
        error_stats["error_rate"] * 100
).round(2)

error_stats.sort_values("error_rate", ascending=False)
print(error_stats)

                                  total  success  errors  error_rate  \
algorithm strategy group                                               
topsis    hedged   adaptive      100000    89177   10823     0.10823   
                   non_adaptive  100000    89224   10776     0.10776   
          none     baseline      100000    89729   10271     0.10271   

                                 error_rate_percent  
algorithm strategy group                             
topsis    hedged   adaptive                   10.82  
                   non_adaptive               10.78  
          none     baseline                   10.27  
